# 09 — Monitoring

In [ ]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

In [ ]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

The monitoring plan: what is tracked per period (score PSI with its chi-square critical value, CSI per attribute, the 12-month early indicator and 24-month performance where observable, calibration slope and intercept), the thresholds, and the decision rule that maps what moved to recalibrate / re-estimate / redevelop. The 2017Q1–2018Q4 vintages are the live monitor range; three simulated drifts show the rule reacting.

Decisions: **T — thresholds and cadence**, **U — the action per scenario**, **V — the monitoring plan text**.

In [ ]:
frame = load_frame(include_indeterminate=True)
mon = A.build_attributes(target.modeling_frame(con, splits=("monitor",), include_indeterminate=True, sample_frac=SAMPLE_FRAC))
early = con.sql("SELECT id, early_bad, early_observable FROM v_split").df()
frame = frame.merge(early, on="id", how="left"); mon = mon.merge(early, on="id", how="left")
card, specs = S.load_card(MODELS / "champion")
for d in (frame, mon):
    sc = S.score(d, card, specs); d["score"] = sc["score"].values; d["eta"] = sc["eta"].values
tr = frame[(frame.split == "train") & frame.bad.notna()]; oot = frame[frame.split == "oot"]
g_dev = V.delong_ci(oot.loc[oot.bad.notna(), "bad"], oot.loc[oot.bad.notna(), "score"])
print("monitor rows", len(mon), "· reference OOT gini %.4f ± %.4f" % (g_dev["gini"], 1.96 * g_dev["gini_se"]))

### Decision T — Thresholds and cadence

PSI 0.10 / 0.25 is the Siddiqi convention; the chi-square critical value scales with n and is the honest test. Gini drop is judged against its DeLong interval; calibration slope against 0.9–1.1 and the intercept-only shift against ±0.15 log-odds (about ±16% in odds). Cadence: monthly PSI, quarterly CSI and early indicator, annual full assessment.

**Your reasoning:**

_(write here)_

In [ ]:
SLOPE_BAND = (0.9, 1.1)
summary = M.monitoring_summary(tr["score"], specs, pd.concat([oot, mon]), g_dev["gini"], 1.96 * g_dev["gini_se"], period_col="vintage_quarter")
for k in ("score_psi", "performance", "calibration", "attribute_drift", "decisions"):
    print(k); display(summary[k])
display(summary["csi"].style.background_gradient(cmap="Reds", axis=None) if hasattr(summary["csi"], "style") else summary["csi"])
for k in ("score_psi", "performance", "calibration", "decisions"): summary[k].to_csv(config.DOCS_DIR / "monitoring" / f"{k}.csv", index=False)
summary["csi"].to_csv(config.DOCS_DIR / "monitoring" / "csi.csv")

In [ ]:
# Evidence — the early indicator by quarter (the only performance signal for censored vintages)
perf = summary["performance"]
fig, ax1 = plt.subplots(figsize=(10, 4)); ax1.plot(perf.vintage_quarter, perf.early_bad_rate, marker="o", label="12-month early bad rate")
ax1.set_ylabel("early bad rate"); ax1.tick_params(axis="x", rotation=60); ax2 = ax1.twinx(); ax2.plot(perf.vintage_quarter, perf.early_gini, marker="s", color="grey", label="early Gini"); ax2.set_ylabel("early Gini")
ax1.legend(loc="upper left"); ax2.legend(loc="upper right"); plt.title("Monitor vintages"); plt.show()

### Decision U — Scenarios

Three simulated drifts on the OOT year: a covariate shift (P(X) moves, P(Y|X) does not), a concept shift (the bad rate rises, ordering intact) and an ordering decay (outcomes shuffled). For each, which rung should the rule land on, and does it?

**Your reasoning:**

_(write here)_

In [ ]:
scenarios = {
    "covariate shift (+0.5 sd on two attributes)": M.simulate_drift(oot, "covariate", attributes=card.attributes[:2], strength=0.5),
    "concept shift (bad rate x1.5, ordering kept)": M.simulate_drift(oot, "concept", strength=0.5, risk_col="eta"),
    "ordering decay (30% outcomes shuffled)": M.simulate_drift(oot, "ordering", strength=0.3),
}
rows = []
for name, d in scenarios.items():
    sc = S.score(d, card, specs); d["score"] = sc["score"].values; d["eta"] = sc["eta"].values
    obs = d[d.bad.notna()]
    p = V.psi(tr["score"], d["score"])["psi"]; g = V.auc_gini_ks(obs["bad"], obs["score"])["gini"]; c = V.calibration_slope_intercept(obs["bad"], obs["eta"])
    mx = max(B.csi(specs[a], d[a])[0] for a in card.attributes)
    rows.append({"scenario": name, **M.decide_action(p, g_dev["gini"], g, 1.96 * g_dev["gini_se"], c["slope"], mx, slope_band=SLOPE_BAND,
                                                     calib_intercept_shift=c["intercept_only_shift"])})
display(pd.DataFrame(rows))

### Decision V — The monitoring plan

Write the plan: metrics, cadence, thresholds, owner, and the action per trigger. This becomes section 8 of `docs/model_document.md`.

**Your plan:**

_(write here)_